In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from scipy.stats import (normaltest, mannwhitneyu, brunnermunzel, kruskal, spearmanr, kendalltau)
from statsmodels.stats.multitest import multipletests

In [2]:
df = pd.read_csv("product_hunt_processed_gp4.csv")
num_cols = [
    "votes_count", "is_featured", "is_ai_topic",
    "topics_count", "max_topic_followers", "avg_topic_followers",
    "tagline_len", "log_votes"
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col])
        
text_cols = ["post_name", "tagline", "primary_topic", "topics_list", "topic_slugs"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)
display(df.head())
print(df.shape)

,post_id,post_name,slug,tagline,votes_count,comments_count,reviews_count,reviews_rating,created_at,featured_at,...,topics_count,max_topic_followers,avg_topic_followers,is_featured,tagline_len,tagline_words,log_votes,topics_text,is_ai_topic,tagline_len_group
0,1151377,Stitch 3.0 by Google,stitch-3-0-by-google,Generate and iterate UI screens with AI on a l...,321,10,0,0.0,2026-05-24 07:01:00+00:00,2026-05-24 07:01:00+00:00,...,3,469204,364965.000000,1,56,11,5.774552,stitch 3 0 by google generate and iterate ui s...,1,41-80
1,1153508,ModelHub,modelhub,The missing menu bar app for local LLMs on Mac.,203,25,0,0.0,2026-05-24 07:01:00+00:00,2026-05-24 07:01:00+00:00,...,4,512870,158689.750000,1,47,10,5.318120,modelhub the missing menu bar app for local ll...,1,41-80
2,1150296,Freu AI,freu-ai-2,Automate any Mac app with $0 recurring run cost,179,13,0,0.0,2026-05-24 07:01:00+00:00,2026-05-24 07:01:00+00:00,...,4,469204,129469.000000,1,47,9,5.192957,freu ai automate any mac app with 0 recurring ...,1,41-80
3,1153432,WhatCable,whatcable,Know what your USB-C cable can really do,162,6,0,0.0,2026-05-24 07:01:00+00:00,2026-05-24 07:01:00+00:00,...,3,15451,13022.666667,1,40,8,5.093750,whatcable know what your usb c cable can reall...,0,0-40
4,1152875,Edgee Fallback Models,edgee-fallback-models,Claude Code that never stops,140,14,2,5.0,2026-05-24 07:01:00+00:00,2026-05-24 07:01:00+00:00,...,3,652259,402537.666667,1,28,5,4.948760,edgee fallback models claude code that never s...,0,0-40


(11339, 26)


Разделим опят на категории проектов

In [3]:
df_featured = df[df["is_featured"] == 1].copy()
df_non_featured = df[df["is_featured"] == 0].copy()

segments = {
    "featured": df_featured,
    "non-featured": df_non_featured
}

display(df["is_featured"].value_counts().rename(index={0: "non-featured", 1: "featured"}))
display(df["is_ai_topic"].value_counts().rename(index={0: "non-AI", 1: "AI"}))

is_featured
non-featured    10673
featured          666
Name: count, dtype: int64

is_ai_topic
non-AI    5973
AI        5366
Name: count, dtype: int64

Перед тестами быстро рассмотрим распределения, чтобы не брать t-test или ANOVA при ненормальном распределении, хотя, в целом, это можно понять по графику из прошлого ноутбука, но лучше будет дополнительно

In [4]:
def get_normal_p(values):
    values = pd.Series(values).dropna()
    if len(values) < 8:
        return np.nan
    return normaltest(values).pvalue
check_rows = []
for segment_name, part in segments.items():
    for col in ["votes_count", "log_votes", "avg_topic_followers", "max_topic_followers", "topics_count", "tagline_len"]:
        values = part[col].dropna()
        check_rows.append({
            "segment": segment_name,
            "column": col,
            "count": len(values),
            "mean": values.mean(),
            "median": values.median(),
            "skew": values.skew(),
            "normal_p": get_normal_p(values)
        })

normal_check = pd.DataFrame(check_rows)
display(normal_check)

,segment,column,count,mean,median,skew,normal_p
0,featured,votes_count,666,129.379880,89.000000,2.845550,4.068045e-91
1,featured,log_votes,666,4.672189,4.499810,1.372432,1.057814e-32
2,featured,avg_topic_followers,666,199154.238013,198326.958333,0.289076,5.224695e-13
3,featured,max_topic_followers,666,378309.249249,469211.000000,-0.538843,0.000000e+00
4,featured,topics_count,666,2.602102,3.000000,-0.552657,1.743488e-13
5,featured,tagline_len,666,46.939940,47.000000,-0.494494,1.097105e-06
6,non-featured,votes_count,10673,2.241544,1.000000,10.669060,0.000000e+00
7,non-featured,log_votes,10673,0.901638,0.693147,0.728919,7.316619e-239
8,non-featured,avg_topic_followers,10673,214219.677646,201074.750000,0.462621,1.079244e-133
9,non-featured,max_topic_followers,10673,399569.910334,469211.000000,-0.474856,0.000000e+00


по votes_count видно, что распределение скошенное, т к среднее и медиана расходятся, асимметрия достаточно заметна, а normal_p маленький, поэтому дальше в основном идут ранговые тесты и перестановочные проверки.здесь их использовать логичнее да и правильнее

In [5]:
test_results = []

def save_result(segment, hypothesis, test, statistic, p_value, effect=np.nan, details=""):
    test_results.append({
        "segment": segment,
        "hypothesis": hypothesis,
        "test": test,
        "statistic": statistic,
        "p_value": p_value,
        "effect": effect,
        "details": details
    })

### 1. AI-тематика и голоса

тут две независимые группы: AI и non-AI. обычный t-test не беру как основной, потому что votes_count не нормальный и с выбросами. основной вариант - Mann-Whitney, он сравнивает группы по рангам. еще беру Brunner-Munzel, потому что он спокойнее относится к разным формам распределений. третья проверка - перестановочный тест по разнице медиан, потому что медиана для таких голосов понятнее среднего

In [6]:
for segment_name, part in segments.items():
    ai_votes = part[part["is_ai_topic"] == 1]["votes_count"].dropna()
    non_ai_votes = part[part["is_ai_topic"] == 0]["votes_count"].dropna()

    print(segment_name)
    print("AI:", len(ai_votes), "медиана:", ai_votes.median())
    print("non-AI:", len(non_ai_votes), "медиана:", non_ai_votes.median())

    u_stat, u_p = mannwhitneyu(ai_votes, non_ai_votes, alternative="two-sided")
    effect = (2 * u_stat) / (len(ai_votes) * len(non_ai_votes)) - 1
    save_result(segment_name,"H1 AI-тематика и голоса","Mann-Whitney",u_stat,u_p,effect,"AI vs non-AI")
    
    bm_stat, bm_p = brunnermunzel(ai_votes, non_ai_votes, alternative="two-sided")
    save_result(segment_name,"H1 AI-тематика и голоса","Brunner-Munzel",bm_stat,bm_p,np.nan,"AI vs non-AI")

    rng = np.random.default_rng(42)
    diff = np.median(ai_votes) - np.median(non_ai_votes)
    full = np.concatenate([ai_votes, non_ai_votes])
    n_ai_votes = len(ai_votes)
    diffs = []
    for _ in range(1000):
        shuffled = rng.permutation(full)
        new_ai_votes = shuffled[:n_ai_votes]
        new_non_ai_votes = shuffled[n_ai_votes:]
        diffs.append(np.median(new_ai_votes) - np.median(new_non_ai_votes))
    diffs = np.array(diffs)
    perm_p = np.mean(np.abs(diffs) >= abs(diff))
    save_result(segment_name,"H1 AI-тематика и голоса","Permutation median",diff,perm_p,diff,"разница медиан AI - non-AI")

featured
AI: 437 медиана: 91.0
non-AI: 229 медиана: 82.0
non-featured
AI: 4929 медиана: 2.0
non-AI: 5744 медиана: 1.0


по этой гипотезе специально взяты три теста. если они дают один и тот же знак, значит результат не держится на одном конкретном методе

### 2. primary_topic и голоса

тут уже больше двух групп, поэтому Mann-Whitney не подходит как общий тест. ANOVA тоже не очень, потому что votes_count скошенный. поэтому берем Kruskal-Wallis - это ранговый аналог ANOVA. категории с совсем маленьким числом продуктов убираем, иначе они будут давать лишний шум

In [7]:
for segment_name, part in segments.items():
    min_count = 20 if segment_name == "featured" else 50
    topic_counts = part["primary_topic"].value_counts()
    good_topics = topic_counts[topic_counts >= min_count].index
    temp = part[part["primary_topic"].isin(good_topics)].copy()
    groups = [temp[temp["primary_topic"] == topic]["votes_count"].dropna()for topic in good_topics]
    print(segment_name)
    print("категорий в тесте:", len(groups))
    print("продуктов в тесте:", len(temp))

    if len(groups) >= 2:
        kw_stat, kw_p = kruskal(*groups)
        n = len(temp)
        k = len(groups)
        epsilon = (kw_stat -k + 1) / (n-k)
        save_result(segment_name,"H2 primary_topic и голоса","Kruskal-Wallis",kw_stat,kw_p,epsilon,"категории с нормальным числом продуктов")
        topic_stat = temp.groupby("primary_topic")["votes_count"].agg(["count", "median"]).sort_values("median", ascending=False)
        display(topic_stat.head(10))

featured
категорий в тесте: 11
продуктов в тесте: 441


,count,median
primary_topic,,
Productivity,101,106.0
Artificial Intelligence,32,100.0
Developer Tools,38,96.5
Open Source,46,96.0
Mac,33,92.0
Design Tools,34,91.5
Chrome Extensions,21,83.0
Pitch Singapore,31,72.0
Pitch Dubai,26,66.0


non-featured
категорий в тесте: 33
продуктов в тесте: 8942


,count,median
primary_topic,,
Mac,78,3.0
API,159,2.0
Chrome Extensions,286,2.0
Fintech,284,2.0
Developer Tools,273,2.0
Hiring,182,2.0
Sales,156,2.0
Productivity,2209,2.0
Open Source,250,2.0


### 3. размер аудитории топиков и голоса

Здесь оба признака числовые, но линейную связи быть не должно, т к рост аудитории топика вряд ли будет прямо пропорционально давать голоса, поэтому основной не Pearson, а Spearman (он смотрит на монотонную связь по рангам)

In [8]:
for segment_name, part in segments.items():
    for col in ["avg_topic_followers", "max_topic_followers"]:
        temp = part[[col, "votes_count"]].dropna()
        corr, p_value = spearmanr(temp[col], temp["votes_count"])
        save_result(segment_name,"H3 аудитория топиков и голоса","Spearman",corr,p_value,corr,col)
        print(segment_name, col)
        print("n:", len(temp), "spearman:", corr, "p-value:", p_value)

featured avg_topic_followers
n: 666 spearman: 0.41009146031934873 p-value: 2.114738159992727e-28
featured max_topic_followers
n: 666 spearman: 0.36456562418524713 p-value: 2.3034857576977327e-22
non-featured avg_topic_followers
n: 10673 spearman: 0.04068667686995564 p-value: 2.615467698755441e-05
non-featured max_topic_followers
n: 10673 spearman: 0.06735633819685233 p-value: 3.2678932536628664e-12


### 4. количество топиков и голоса

topics_count непрерывная величина, но не в классическом понимании, т к там небольшие целые числа, поэтому снова берем Spearman. он нормально подходит для такой порядковой связи (от увеличения количества топиков смотрим будет больше/меньше голосов)

In [9]:
for segment_name, part in segments.items():
    temp = part[["topics_count", "votes_count"]].dropna()
    corr, p_value = spearmanr(temp["topics_count"], temp["votes_count"])
    save_result(segment_name,"H4 количество топиков и голоса","Spearman",corr,p_value,corr,"topics_count")
    display(part.groupby("topics_count")["votes_count"].agg(["count", "median"]))
    print(segment_name)
    print("spearman:", corr, "p-value:", p_value)

,count,median
topics_count,,
1,153,67.0
2,44,107.0
3,388,98.5
4,77,95.0
5,4,139.5


featured
spearman: 0.40009265731204563 p-value: 5.409688649289816e-27


,count,median
topics_count,,
1,1127,1.0
2,1173,1.0
3,7298,2.0
4,1038,2.0
5,36,2.0
6,1,2.0


non-featured
spearman: 0.15476330693193208 p-value: 3.31690113172061e-58


### 5. длина tagline и голоса

тут tagline берем по количеству символаов. связь может быть слабой и нелинейной, поэтому основной тест у нас Spearman, а для сравнения Kendall tau (тоже ранговый, но считает связь чуть строже) и третья проверка это перестановочный Spearman, чтобы проверить, не получилась ли корреляция случайно

In [10]:
for segment_name, part in segments.items():
    temp = part[["tagline_len", "votes_count"]].dropna()
    sp_corr, sp_p = spearmanr(temp["tagline_len"], temp["votes_count"])
    save_result(segment_name,"H5 длина tagline и голоса","Spearman",sp_corr,sp_p,sp_corr,"tagline_len")
    
    kd_corr, kd_p = kendalltau(temp["tagline_len"], temp["votes_count"])
    save_result(segment_name,"H5 длина tagline и голоса","Kendall tau",kd_corr,kd_p,kd_corr,"tagline_len")

    rng = np.random.default_rng(42)
    perm_corr = spearmanr(temp["tagline_len"], temp["votes_count"]).statistic
    corr_values = []
    for _ in range(1000):
        peremesh = rng.permutation(temp["tagline_len"])
        corr_values.append(spearmanr(peremesh, temp["votes_count"]).statistic)
    corr_values = np.array(corr_values)
    perm_p = np.mean(np.abs(corr_values) >= abs(perm_corr))
    save_result(segment_name,"H5 длина tagline и голоса","Permutation Spearman",perm_corr,perm_p,perm_corr,"tagline_len")

    print(segment_name)
    print("spearman:", sp_corr, "p-value:", sp_p)
    print("kendall:", kd_corr, "p-value:", kd_p)
    print("permutation:", perm_corr, "p-value:", perm_p)

featured
spearman: 0.11417579189363415 p-value: 0.0031710487162201623
kendall: 0.07737769954293754 p-value: 0.003454617122988685
permutation: 0.11417579189363415 p-value: 0.003
non-featured
spearman: 0.0426006540431815 p-value: 1.0697818771242388e-05
kendall: 0.03135173432231946 p-value: 1.2168915723877478e-05
permutation: 0.0426006540431815 p-value: 0.0


по этой гипотезе тоже три теста. если Spearman, Kendall и перестановочная проверка сходятся, так что результат один, и вывод будет прекрасно изложен в презентации)) но всё же кратко: если p-value маленький, но корреляция около нуля, то это не сильный фактор, а просто слабая статистическая связь

СОбираем все тесты в одну таблицу и поправляем p-value, потому что проверок много

In [11]:
results = pd.DataFrame(test_results)
results["p_holm"] = np.nan
results["significant_holm"] = False

for segment_name in results["segment"].unique():
    mask = (results["segment"] == segment_name) & (results["p_value"].notna())
    reject, p_holm, _, _ = multipletests(results.loc[mask, "p_value"],alpha=0.05,method="holm")
    results.loc[mask, "p_holm"] = p_holm
    results.loc[mask, "significant_holm"] = reject

display(results.sort_values(["hypothesis", "segment", "test"]))

,segment,hypothesis,test,statistic,p_value,effect,details,p_holm,significant_holm
1,featured,H1 AI-тематика и голоса,Brunner-Munzel,-2.056532e+00,4.020481e-02,NaN,AI vs non-AI,1.206144e-01,False
0,featured,H1 AI-тематика и голоса,Mann-Whitney,5.473150e+04,4.651562e-02,0.093832,AI vs non-AI,1.206144e-01,False
2,featured,H1 AI-тематика и голоса,Permutation median,9.000000e+00,4.200000e-02,9.000000,разница медиан AI - non-AI,1.206144e-01,False
4,non-featured,H1 AI-тематика и голоса,Brunner-Munzel,-8.163922e+00,3.626525e-16,NaN,AI vs non-AI,2.538567e-15,True
3,non-featured,H1 AI-тематика и голоса,Mann-Whitney,1.541562e+07,3.739984e-16,0.088975,AI vs non-AI,2.538567e-15,True
5,non-featured,H1 AI-тематика и голоса,Permutation median,1.000000e+00,1.000000e-03,1.000000,разница медиан AI - non-AI,1.000000e-03,True
6,featured,H2 primary_topic и голоса,Kruskal-Wallis,1.715161e+02,1.346270e-31,0.375619,категории с нормальным числом продуктов,1.346270e-30,True
7,non-featured,H2 primary_topic и голоса,Kruskal-Wallis,1.630011e+02,1.748173e-19,0.014704,категории с нормальным числом продуктов,1.398538e-18,True
8,featured,H3 аудитория топиков и голоса,Spearman,4.100915e-01,2.114738e-28,0.410091,avg_topic_followers,1.903264e-27,True
9,featured,H3 аудитория топиков и голоса,Spearman,3.645656e-01,2.303486e-22,0.364566,max_topic_followers,1.612440e-21,True


In [12]:
summary_rows = []
for _, row in results.iterrows():
    if pd.isna(row["p_holm"]):
        decision = "нет p-value"
    elif row["p_holm"] < 0.05:
        decision = "значимо"
    else:
        decision = "не значимо"
    summary_rows.append({"segment": row["segment"],"hypothesis": row["hypothesis"],"test": row["test"],"p_holm": round(row["p_holm"], 5) if not pd.isna(row["p_holm"]) else None,"effect": round(row["effect"], 5) if not pd.isna(row["effect"]) else None,"decision": decision})

summary_table = pd.DataFrame(summary_rows)
display(summary_table)

,segment,hypothesis,test,p_holm,effect,decision
0,featured,H1 AI-тематика и голоса,Mann-Whitney,0.12061,0.09383,не значимо
1,featured,H1 AI-тематика и голоса,Brunner-Munzel,0.12061,NaN,не значимо
2,featured,H1 AI-тематика и голоса,Permutation median,0.12061,9.00000,не значимо
3,non-featured,H1 AI-тематика и голоса,Mann-Whitney,0.00000,0.08897,значимо
4,non-featured,H1 AI-тематика и голоса,Brunner-Munzel,0.00000,NaN,значимо
5,non-featured,H1 AI-тематика и голоса,Permutation median,0.00100,1.00000,значимо
6,featured,H2 primary_topic и голоса,Kruskal-Wallis,0.00000,0.37562,значимо
7,non-featured,H2 primary_topic и голоса,Kruskal-Wallis,0.00000,0.01470,значимо
8,featured,H3 аудитория топиков и голоса,Spearman,0.00000,0.41009,значимо
9,featured,H3 аудитория топиков и голоса,Spearman,0.00000,0.36457,значимо
